# 05 · Comparaciones que ayudan a decidir

**RA2 · CE2.1–2.4.** Formular H0, interpretar efectos e intervalos, comparar medias/proporciones y controlar multiplicidad. Dos sesiones y práctica.

**Caso:** ComercioSur, datos simulados. Ejecutar en orden con kernel activo. Cada ejercicio exige una interpretación y un límite; consultar [manual](../material_propio/EBA_manual_cientifico.html) y [guía](../guia_maestra_eba.html). El curso presupone manejo elemental de tablas de Fundamentos. No instalar paquetes desde las celdas.

### Antes de ejecutar

Núcleo · 90–120 min orientativos.

**Objetivo:** Elegir un contraste y comunicar efecto, incertidumbre y relevancia práctica.

**Preparación:** Intervalos y diseño independiente o pareado.

**Ejemplo de referencia:** Comparar campañas con Welch y ANOVA; usar Holm cuando se examinan varios pares.

**Práctica:** Cambiar costo del tratamiento sin modificar p y revisar si cambia la recomendación.

**Criterio de logro:** Entregar diferencia, IC, p, ajuste por multiplicidad y una decisión condicionada.

[Guía y secuencia](../guia_maestra_eba.html#ruta-aprendizaje) · [Fundamento del manual](../material_propio/EBA_manual_cientifico.html#sec-contrastes)

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, Markdown
RAIZ=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'_transversal/datos').is_dir())
sys.path.insert(0,str(RAIZ/'02_Estadistica_Business_Analytics/reproducibilidad'))
from eba_recursos import *
df=pedidos()
plt.rcParams.update({'figure.figsize':(7,3.4),'axes.spines.top':False,'axes.spines.right':False})


<a id="hipotesis"></a>
## Una media, antes de mirar p

H0: media del tiempo=35 minutos frente a diferencia bilateral; α=.05 fijado antes. El valor p se calcula bajo H0 y supuestos; no mide P(H0|datos), ni tamaño de efecto. No rechazo no demuestra equivalencia.

In [2]:
r=stats.ttest_1samp(df.tiempo_min,35);print('Diferencia vs 35:',df.tiempo_min.mean()-35,'min; p:',r.pvalue);display(ic_media(df.tiempo_min))

Diferencia vs 35: 1.2562833333333359 min; p: 9.450809733537722e-05


{'n': 600,
 'media': np.float64(36.256283333333336),
 'sd': np.float64(7.828359217173848),
 'se': np.float64(0.3195914267548248),
 'li': np.float64(35.6286274238431),
 'ls': np.float64(36.88393924282357)}

<a id="welch"></a>
## Dos grupos independientes: B−A

Se fija tiempo como resultado primario. Welch permite varianzas diferentes. La diferencia se interpreta con su IC y un beneficio mínimo de 3 minutos definido para el ejemplo. Si el IC cruza −3, la evidencia sobre ese beneficio mínimo no es concluyente aunque excluya cero.

In [3]:
a=df.loc[df.campana.eq('A'),'tiempo_min'];b=df.loc[df.campana.eq('B'),'tiempo_min'];display(pd.Series(welch(b,a)));print('Umbral práctico predefinido: diferencia <= -3 minutos')

diferencia   -4.068278e+00
t            -5.940487e+00
gl            4.154185e+02
p             6.014918e-09
li           -5.414460e+00
ls           -2.722096e+00
dtype: float64

Umbral práctico predefinido: diferencia <= -3 minutos


<a id="pareado"></a>
## Antes/después no son grupos independientes

Contraste t de las diferencias de las mismas 80 personas. No hay control concurrente: el cambio puede reflejar tiempo, aprendizaje u otros factores. Un p pequeño no elimina esas explicaciones.

In [4]:
pares=pd.read_csv(DATOS/'comerciosur_pareado.csv');dif=pares.despues-pares.antes;print(stats.ttest_rel(pares.despues,pares.antes));display(ic_media(dif));assert np.isclose(stats.ttest_rel(pares.despues,pares.antes).statistic,stats.ttest_1samp(dif,0).statistic)

TtestResult(statistic=np.float64(-5.833140897696923), pvalue=np.float64(1.1390776550591756e-07), df=np.int64(79))


{'n': 80,
 'media': np.float64(-2.414125),
 'sd': np.float64(3.70170904550789),
 'se': np.float64(0.41386365293407523),
 'li': np.float64(-3.237899994989239),
 'ls': np.float64(-1.5903500050107608)}

<a id="proporciones"></a>
## Una y dos proporciones

Para A, contraste binomial exacto H0:p=.70. Para B−A, aproximación normal bilateral usa proporción combinada bajo H0 y requiere suficientes éxitos/fracasos. El IC Wald de diferencia se muestra como aproximación para estos conteos, no como receta para muestras escasas.

In [5]:
aa=df[df.campana.eq('A')].resuelto;bb=df[df.campana.eq('B')].resuelto;print('Binomial exacto A vs .70:',stats.binomtest(int(aa.sum()),len(aa),.7).pvalue);pooled=(aa.sum()+bb.sum())/(len(aa)+len(bb));se0=np.sqrt(pooled*(1-pooled)*(1/len(aa)+1/len(bb)));delta=bb.mean()-aa.mean();z=delta/se0;se=np.sqrt(aa.mean()*(1-aa.mean())/len(aa)+bb.mean()*(1-bb.mean())/len(bb));print('B-A (puntos porcentuales):',100*delta,'p bilateral:',2*stats.norm.sf(abs(z)),'IC aprox:',100*np.array([delta-1.96*se,delta+1.96*se]));assert min(aa.sum(),len(aa)-aa.sum(),bb.sum(),len(bb)-bb.sum())>=10

Binomial exacto A vs .70: 0.5459675271905724
B-A (puntos porcentuales): 8.61244019138756 p bilateral: 0.049275257309657566 IC aprox: [ 0.06696065 17.15791973]


<a id="anova"></a>
## Tres campañas: contraste global y magnitud

H0: μA=μB=μC. ANOVA clásico compara variación entre y dentro de grupos con varianzas comunes; Welch es alternativa para heterocedasticidad. Eta² es proporción descriptiva de variación explicada por grupos, no porcentaje causal universal. Rechazo global no identifica qué pares difieren.

In [6]:
display(pd.Series(anova(df)));print('Levene centrado en mediana:',stats.levene(*(g.tiempo_min for _,g in df.groupby('campana')),center='median'));display(comparar_medias(df));print('Los IC por par son individuales 95%; p_Holm controla error familiar para los tres contrastes, no convierte esos IC en simultáneos.')

F          6.301870e+01
p          1.475196e-25
F_Welch    6.267293e+01
p_Welch    2.419560e-24
eta2       1.743166e-01
dtype: float64

Levene centrado en mediana: LeveneResult(statistic=np.float64(0.7866591787163867), pvalue=np.float64(0.45583497784060734))


,contraste,diferencia,t,gl,p,li,ls,p_Holm
0,B - A,-4.068278,-5.940487,415.418546,6.014918e-09,-5.414460,-2.722096,1.202984e-08
1,C - A,-8.102625,-11.162512,372.101899,3.765548e-25,-9.529963,-6.675287,1.129664e-24
2,C - B,-4.034347,-5.467590,377.290407,8.314451e-08,-5.485192,-2.583503,8.314451e-08


Los IC por par son individuales 95%; p_Holm controla error familiar para los tres contrastes, no convierte esos IC en simultáneos.


<a id="error-potencia"></a>
## Error tipo I, tipo II y potencia

Simulación normal de dos grupos independientes con SD=7. El umbral p<.05 detecta diferencias de medias; potencia depende de efecto, n y dispersión. Actividad: compare efecto=0 (error tipo I) y efecto=3 con n=20/100; explique por qué no se debe calcular potencia observada para justificar un resultado.

In [7]:
@widgets.interact(n=widgets.IntSlider(value=50,min=20,max=200,step=20),efecto=widgets.FloatSlider(value=3,min=0,max=6,step=.5))
def potencia(n,efecto):
    rng=np.random.default_rng(SEMILLA);a=rng.normal(35,7,(1000,n));b=rng.normal(35-efecto,7,(1000,n));p=stats.ttest_ind(b,a,axis=1,equal_var=False).pvalue;print(f'Fracción de rechazos: {(p<.05).mean():.3f}; efecto simulado {efecto:.1f} min; n por grupo {n}')

interactive(children=(IntSlider(value=50, description='n', max=200, min=20, step=20), FloatSlider(value=3.0, d…

<a id="decision"></a>
## Actividad: recomendación condicionada

Decida si avanzar a piloto considerando reducción mínima, costo por pedido, incertidumbre y resolución. Distinguir resultado primario de exploraciones secundarias. No buscar repetidamente segmentos con p<.05 sin control de multiplicidad.

In [8]:
@widgets.interact(costo=widgets.FloatSlider(value=.5,min=0,max=2,step=.1),valor_minuto=widgets.FloatSlider(value=.2,min=.05,max=.5,step=.05))
def negocio(costo,valor_minuto):
    r=welch(b,a);ahorro=-r['diferencia']*valor_minuto-costo;li=-r['ls']*valor_minuto-costo;ls=-r['li']*valor_minuto-costo;print(f'Ahorro estimado B vs A: {ahorro:.3f} UM/pedido; IC transformado [{li:.3f}, {ls:.3f}]. Costos tratados como constantes docentes.')

interactive(children=(FloatSlider(value=0.5, description='costo', max=2.0), FloatSlider(value=0.2, description…

## Comprobación y entrega

Responda la autoevaluación y complete la actividad del último bloque. Entregue objetivo, método, supuesto, resultado con unidades y decisión condicionada. Las soluciones orientadoras permiten estudiar, pero no sustituyen su interpretación.

In [9]:
auto_05=pregunta('¿ANOVA significativo identifica automáticamente la mejor campaña?','Sí, basta con su valor p','No, requiere contrastes, magnitud, supuestos y costos','El contraste global no determina qué medias difieren ni cuál alternativa conviene.')
display(auto_05)